# SpecFreak — Phase 2: TF-IDF + Cosine Similarity
### NLP-Based Game Recommendation Engine
**What this phase does:**
- Combines game description + genre + platform + type into one text feature
- Converts that text into numbers using TF-IDF
- Takes your natural language prompt and finds the most similar games using Cosine Similarity
- Returns top N game recommendations

## Step 1 — Install & Import Libraries

In [ ]:
# Run this cell first to make sure all libraries are available
# In Google Colab these are already pre-installed, so this should run fine

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re

print("All libraries imported successfully!")

All libraries imported successfully!


## Step 2 — Load the Dataset

In [ ]:
# ---------------------------------------------------------
# IMPORTANT: Make sure your CSV file is uploaded to Colab
# Go to the Files panel on the left → Upload
# Upload: games_of_all_time_cleaned_normalized.csv
# ---------------------------------------------------------

df = pd.read_csv('games_of_all_time_cleaned_normalized.csv')

print(f"Dataset loaded: {df.shape[0]} games, {df.shape[1]} columns")
print("\nColumns available:")
print(df.columns.tolist())
print("\nFirst 3 games:")
df[['game_name', 'genre', 'platform', 'type', 'meta_score']].head(3)

Dataset loaded: 8831 games, 12 columns

Columns available:
['game_name', 'meta_score', 'user_score', 'platform', 'description', 'url', 'developer', 'genre', 'type', 'rating', 'meta_score_norm_0_1', 'user_score_norm_0_1']

First 3 games:


,game_name,genre,platform,type,meta_score
0,The Legend of Zelda: Ocarina of Time,"['Action Adventure', 'Fantasy']",['nintendo-64'],singleplayer,99.0
1,Super Mario Galaxy,"['Action', 'Platformer', '3D']",['wii'],singleplayer,97.0
2,Super Mario Galaxy 2,"['Action', 'Platformer', '3D']",['wii'],singleplayer,97.0


## Step 3 — Feature Engineering (Combine Text Columns)

In [ ]:
# ------------------------------------------------------------------
# WHAT WE ARE DOING HERE:
# TF-IDF works on a single text column.
# So we COMBINE: description + genre + platform + type
# into one big text string called 'combined_features'.
# This gives the model more context about each game.
# ------------------------------------------------------------------

def clean_list_string(text):
    """
    Genre and platform are stored as strings like:
    "['Action Adventure', 'Fantasy']"
    This function converts that into plain text:
    "Action Adventure Fantasy"
    """
    if pd.isna(text):
        return ''
    # Remove brackets, quotes, commas
    text = re.sub(r"[\[\]'\"]", '', str(text))
    text = re.sub(r',', ' ', text)
    return text.strip().lower()

def clean_text(text):
    """Basic text cleaning for description and type fields."""
    if pd.isna(text):
        return ''
    return str(text).strip().lower()


# Apply cleaning
df['clean_genre']    = df['genre'].apply(clean_list_string)
df['clean_platform'] = df['platform'].apply(clean_list_string)
df['clean_type']     = df['type'].apply(clean_text)
df['clean_desc']     = df['description'].apply(clean_text)

# Combine all into ONE text field
# We repeat genre and type 2x to give them more weight in TF-IDF
df['combined_features'] = (
    df['clean_desc'] + ' ' +
    df['clean_genre'] + ' ' + df['clean_genre'] + ' ' +   # genre repeated for emphasis
    df['clean_platform'] + ' ' +
    df['clean_type'] + ' ' + df['clean_type']              # type repeated for emphasis
)

print("Feature engineering done!")
print("\nSample combined feature for game 1:")
print(df['combined_features'].iloc[0][:300], '...')

Feature engineering done!

Sample combined feature for game 1:
as a young boy, link is tricked by ganondorf, the king of the gerudo thieves. the evil human uses link to gain access to the sacred realm, where he places his tainted hands on triforce and transforms the beautiful hyrulean landscape into a barren wasteland. link is determined to fix the problems he  ...


## Step 4 — Build the TF-IDF Matrix

In [ ]:
# ------------------------------------------------------------------
# WHAT IS TF-IDF?
# TF  = Term Frequency   → how often a word appears in THIS game's text
# IDF = Inverse Document Frequency → how rare that word is across ALL games
# TF-IDF score = high if the word is COMMON in this game but RARE overall
# Result: Each game becomes a vector of numbers (one number per unique word)
# ------------------------------------------------------------------

tfidf = TfidfVectorizer(
    max_features=10000,    # use top 10,000 most important words
    stop_words='english',  # remove common words like 'the', 'is', 'and'
    ngram_range=(1, 2),    # consider single words AND 2-word phrases (e.g. 'open world')
    min_df=2               # ignore words that appear in less than 2 games
)

print("Building TF-IDF matrix... (this may take 10-20 seconds)")
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f"\nTF-IDF matrix built successfully!")
print(f"Matrix shape: {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} games")
print(f"  → {tfidf_matrix.shape[1]} unique words/phrases")

Building TF-IDF matrix... (this may take 10-20 seconds)

TF-IDF matrix built successfully!
Matrix shape: (8831, 10000)
  → 8831 games
  → 10000 unique words/phrases


## Step 5 — The Recommendation Function

In [ ]:
# ------------------------------------------------------------------
# WHAT IS COSINE SIMILARITY?
# It measures the angle between two vectors.
# Score = 1.0  → exact same direction = very similar
# Score = 0.0  → completely different
#
# HOW WE USE IT:
# 1. Take the user's text prompt
# 2. Convert it to a TF-IDF vector using the SAME vectorizer
# 3. Compare it against ALL 8831 game vectors
# 4. Return games with highest similarity score
# ------------------------------------------------------------------

def recommend_games(user_prompt, top_n=10):
    """
    Takes a natural language prompt from the user
    and returns the top_n most similar games.

    Parameters:
        user_prompt (str) : e.g. "dark story driven RPG with open world"
        top_n (int)       : how many recommendations to return (default 10)

    Returns:
        DataFrame with recommended games and their similarity scores
    """

    # Step A: Convert user prompt to TF-IDF vector
    user_vector = tfidf.transform([user_prompt.lower()])

    # Step B: Compute cosine similarity between user vector and ALL games
    similarity_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    # Step C: Get indices of top N most similar games
    top_indices = similarity_scores.argsort()[::-1][:top_n]

    # Step D: Build results dataframe
    results = df.iloc[top_indices][[
        'game_name', 'genre', 'platform', 'type',
        'meta_score', 'user_score', 'rating'
    ]].copy()

    results['similarity_score'] = similarity_scores[top_indices].round(4)
    results = results.reset_index(drop=True)
    results.index += 1  # Start ranking from 1

    return results


print("Recommendation function is ready!")
print("Now go to the next cell and test it.")

Recommendation function is ready!
Now go to the next cell and test it.


## Step 6 — Test the Recommender ✅

In [ ]:
# ------------------------------------------------------------------
# CHANGE THE PROMPT BELOW TO ANYTHING YOU WANT AND RUN THIS CELL
# ------------------------------------------------------------------

user_prompt = "dark story driven RPG with open world exploration and rich narrative"

print(f"User Prompt: '{user_prompt}'")
print("=" * 70)

results = recommend_games(user_prompt, top_n=10)
print(results.to_string())

User Prompt: 'dark story driven RPG with open world exploration and rich narrative'
                                      game_name                                                                      genre                                       platform          type  meta_score  user_score rating  similarity_score
1                       Kena: Bridge of Spirits                              ['Action Adventure', 'General', 'Open-World']                        ['pc', 'playstation-5']  singleplayer       81.50        81.5      T            0.3501
2                            Shape of the World                                         ['Action Adventure', 'Open-World']                              ['playstation-4']  singleplayer       72.00        72.0      E            0.2664
3   The Witcher 3: Wild Hunt - Complete Edition                                             ['Role-Playing', 'Action RPG']                                     ['switch']  singleplayer       85.00        85.0      M   

In [ ]:
# Test with a different prompt
user_prompt2 = "old school platformer with coins and jumping"

print(f"User Prompt: '{user_prompt2}'")
print("=" * 70)

results2 = recommend_games(user_prompt2, top_n=10)
print(results2.to_string())

User Prompt: 'old school platformer with coins and jumping'
                                    game_name                                      genre                       platform          type  meta_score  user_score rating  similarity_score
1                             Mega Coin Squad             ['Action', 'Platformer', '2D']                   ['xbox-one']  singleplayer        74.0        64.0      T            0.2643
2                                 JumpJet Rex             ['Action', 'Platformer', '2D']                         ['pc']  singleplayer        76.0        70.0      T            0.2533
3                                   SwapQuest                      ['Action', 'General']           ['playstation-vita']  singleplayer        64.0        70.0   E10+            0.2488
4                       Slain: Back from Hell             ['Action', 'Platformer', '2D']                         ['pc']  singleplayer        74.0        81.0      T            0.2442
5                        

In [ ]:
# Test with another prompt
user_prompt3 = "relaxing puzzle game suitable for kids with cute graphics"

print(f"User Prompt: '{user_prompt3}'")
print("=" * 70)

results3 = recommend_games(user_prompt3, top_n=10)
print(results3.to_string())

User Prompt: 'relaxing puzzle game suitable for kids with cute graphics'
                         game_name                                   genre           platform          type  meta_score  user_score rating  similarity_score
1                 Puzzle Dimension  ['Miscellaneous', 'Puzzle', 'General']  ['playstation-3']  singleplayer        79.0        79.0      E            0.2377
2                      Pony Island                   ['Puzzle', 'General']             ['pc']  singleplayer        86.0        74.0      T            0.2101
3                         Maquette                   ['Puzzle', 'General']  ['playstation-5']  singleplayer        70.0        63.0      T            0.2099
4                             RUSH  ['Miscellaneous', 'Puzzle', 'General']          ['wii-u']  singleplayer        77.0        75.0      E            0.2083
5                     Room to Grow                   ['Puzzle', 'General']             ['pc']  singleplayer        76.0        76.0      T    

## Step 7 — Interactive Mode (Type Your Own Prompt)

In [ ]:
# ------------------------------------------------------------------
# Run this cell to enter your own prompt live in Colab
# ------------------------------------------------------------------

while True:
    prompt = input("\nDescribe the game you want (or type 'quit' to stop): ")

    if prompt.lower() == 'quit':
        print("Exiting recommendation mode.")
        break

    if len(prompt.strip()) < 3:
        print("Please enter a longer description.")
        continue

    print(f"\nTop 10 games for: '{prompt}'")
    print("-" * 60)
    res = recommend_games(prompt, top_n=20)
    print(res[['game_name', 'genre', 'type', 'meta_score', 'similarity_score']].to_string())


Describe the game you want (or type 'quit' to stop): old school platformer with coins and jumping

Top 10 games for: 'old school platformer with coins and jumping'
------------------------------------------------------------
                                    game_name                                                          genre          type  meta_score  similarity_score
1                             Mega Coin Squad                                 ['Action', 'Platformer', '2D']  singleplayer       74.00            0.3204
2   The Witcher 3: Wild Hunt - Blood and Wine                                 ['Role-Playing', 'Action RPG']  singleplayer       91.50            0.1961
3                                   SwapQuest                                          ['Action', 'General']  singleplayer       64.00            0.1949
4            Danganronpa: Trigger Happy Havoc                                  ['Adventure', 'Visual Novel']  singleplayer       81.00            0.1850
5        

## Phase 2 Complete ✅

**What was built in this phase:**

| Step | What happened |
|------|---------------|
| Feature Engineering | Combined description + genre + platform + type into one text |
| TF-IDF Vectorizer | Converted 8,831 game texts into numerical vectors (10,000 features) |
| Cosine Similarity | Measures how close a user prompt is to each game vector |
| Recommender Function | Returns top N games ranked by similarity score |

**Next → Phase 3:** Add sentiment scoring using meta_score_norm_0_1 + user_score_norm_0_1 to improve ranking quality.

NOW GETTING ON SENTIMENT


In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("games_of_all_time_cleaned_normalized.csv")

In [ ]:
df.columns

Index(['game_name', 'meta_score', 'user_score', 'platform', 'description',
       'url', 'developer', 'genre', 'type', 'rating', 'meta_score_norm_0_1',
       'user_score_norm_0_1'],
      dtype='object')

In [ ]:
df['meta_score'] = pd.to_numeric(df['meta_score'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')

In [ ]:
df['meta_norm'] = df['meta_score'] / 100
df['user_norm'] = df['user_score'] / 10

In [ ]:
def compute_sentiment(meta, user):
    if pd.notna(meta) and pd.notna(user):
        return 0.6 * (meta / 100) + 0.4 * (user / 10)

    elif pd.notna(meta):
        return meta / 100

    elif pd.notna(user):
        return user / 10

    else:
        return 0

In [ ]:
df['sentiment_score'] = df.apply(
    lambda row: compute_sentiment(row['meta_score'], row['user_score']),
    axis=1
)

In [ ]:
df[['meta_score', 'user_score', 'sentiment_score']].head(10)

,meta_score,user_score,sentiment_score
0,99.0,91.0,4.234
1,97.0,91.0,4.222
2,97.0,91.0,4.222
3,97.0,89.0,4.142
4,97.0,89.0,4.142
5,97.0,87.0,4.062
6,97.0,83.0,3.902
7,97.0,62.0,3.062
8,96.0,88.0,4.096
9,96.0,91.0,4.216


In [ ]:
df['sentiment_score'].describe()

,sentiment_score
count,8831.000000
mean,3.221416
std,0.572186
min,0.458000
25%,2.923000
50%,3.334000
75%,3.628000
max,4.354000


In [ ]:
df.to_csv("games_with_sentiment.csv", index=False)

**NOW STEP 4 MAKING hybrid scroing SYSTEM**

In [ ]:
df = pd.read_csv("games_with_sentiment.csv")
df.head()

,game_name,meta_score,user_score,platform,description,url,developer,genre,type,rating,meta_score_norm_0_1,user_score_norm_0_1,meta_norm,user_norm,sentiment_score
0,The Legend of Zelda: Ocarina of Time,99.0,91.0,['nintendo-64'],"As a young boy, Link is tricked by Ganondorf, ...",https://www.metacritic.com/game/nintendo-64/th...,Nintendo,"['Action Adventure', 'Fantasy']",singleplayer,E,1.000000,0.927083,0.99,9.1,4.234
1,Super Mario Galaxy,97.0,91.0,['wii'],[Metacritic's 2007 Wii Game of the Year] The u...,https://www.metacritic.com/game/wii/super-mari...,Nintendo,"['Action', 'Platformer', '3D']",singleplayer,E,0.977273,0.927083,0.97,9.1,4.222
2,Super Mario Galaxy 2,97.0,91.0,['wii'],"Super Mario Galaxy 2, the sequel to the galaxy...",https://www.metacritic.com/game/wii/super-mari...,Nintendo EAD Tokyo,"['Action', 'Platformer', '3D']",singleplayer,E,0.977273,0.927083,0.97,9.1,4.222
3,Metroid Prime,97.0,89.0,['gamecube'],Samus returns in a new mission to unravel the ...,https://www.metacritic.com/game/gamecube/metro...,Retro Studios,"['Action', 'Shooter', 'First-Person', 'Sci-Fi']",singleplayer,T,0.977273,0.906250,0.97,8.9,4.142
4,Super Mario Odyssey,97.0,89.0,['switch'],New Evolution of Mario Sandbox-Style Gameplay....,https://www.metacritic.com/game/switch/super-m...,Nintendo,"['Action', 'Platformer', '3D']",singleplayer,E10+,0.977273,0.906250,0.97,8.9,4.142


In [ ]:
df = pd.read_csv("games_with_sentiment.csv")

In [ ]:
print(df.columns)

Index(['game_name', 'meta_score', 'user_score', 'platform', 'description',
       'url', 'developer', 'genre', 'type', 'rating', 'meta_score_norm_0_1',
       'user_score_norm_0_1', 'meta_norm', 'user_norm', 'sentiment_score'],
      dtype='object')


In [ ]:
df.rename(columns={'name': 'title'}, inplace=True)

In [ ]:
print(df.columns)

Index(['game_name', 'meta_score', 'user_score', 'platform', 'description',
       'url', 'developer', 'genre', 'type', 'rating', 'meta_score_norm_0_1',
       'user_score_norm_0_1', 'meta_norm', 'user_norm', 'sentiment_score'],
      dtype='object')


In [ ]:
tfidf_matrix = tfidf.fit_transform(df['description'].fillna(""))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['description'].fillna(""))

In [ ]:
alpha = 0.7
beta = 0.3
final_score = alpha * similarity_scores + beta * sentiment

NameError: name 'similarity_scores' is not defined

In [ ]:
def recommend_games(user_input, df, tfidf, tfidf_matrix, top_n=5):

    # Convert user input into vector
    user_vec = tfidf.transform([user_input])

    # Compute similarity
    similarity_scores = cosine_similarity(user_vec, tfidf_matrix).flatten()

    # Add similarity to dataframe
    df['similarity'] = similarity_scores

    # Hybrid score
    alpha = 0.7
    beta = 0.3

    df['final_score'] = (alpha * df['similarity']) + (beta * df['sentiment_score'])

    # Sort results
    results = df.sort_values(by='final_score', ascending=False)

    return results[['title', 'similarity', 'sentiment_score', 'final_score']].head(top_n)

In [ ]:
recommend_games("old school platformer with coins and jumping", df, tfidf, tfidf_matrix)